# Question selection 

A random sample contains few questions the models already know (about 1 in 4), and even fewer survive the later filters.
So the final questions are chosen in three stages:

1. **Closed-book on the whole validation set** 7,405 questions, both models.
2. **Poisonable only:** bridge questions whose answer is stated in exactly one gold paragraph that is not about the answer itself (rules only, no model).
3. **Stratified sample:** 400 questions **known by at least one model** + 400 **known by neither**, sampled separately.

Because the two groups are sampled separately, the final sample does not have the natural mix of known and unknown questions.

The selection is saved to `data/poison/final_selection.json`.
notebook 02 with `RUN = "final"` generates the poison, and notebooks 01, 03, 04 with `RUN = "final"` run the experiments.

In [1]:
import json
from pathlib import Path

import pandas as pd

from ragtrust.analysis import known_ids
from ragtrust.data import load_hotpotqa, pick_fewshot_examples
from ragtrust.experiment import cached_run, run_setting
from ragtrust.llm import OllamaLLM
from ragtrust.poison import find_answer_passage
from ragtrust.runs import SELECTION_PATH, stratified_sample

MODELS = ["llama3.1:8b", "qwen2.5:7b"]
N_KNOWN, N_UNKNOWN = 400, 400
SEED = 0
RESULTS = Path("../results")

all_examples = load_hotpotqa(n=None)
print(f"Validation questions: {len(all_examples)}")

Validation questions: 7405


## 1. Closed-book on everything

About 7,400 answers per model. 

In [2]:
fewshot = pick_fewshot_examples(seed=SEED)
frames = []
for model in MODELS:
    llm = OllamaLLM(model=model, seed=SEED)
    path = RESULTS / f"selection_closed_book_{model.replace(':', '-')}_all.csv"
    run = lambda: run_setting(llm, all_examples, "closed_book", fewshot)
    frames.append(cached_run(path, llm, run).assign(model=model))
closed_book = pd.concat(frames)

known = {m: known_ids(closed_book, m) for m in MODELS}
for m in MODELS:
    print(f"{m}: known {len(known[m])} of {len(all_examples)} ({len(known[m]) / len(all_examples):.1%})")

closed_book: 100%|██████████| 7405/7405 [13:39<00:00,  9.04it/s]


llama3.1:8b: known 1721 of 7405 (23.2%)
qwen2.5:7b: known 1531 of 7405 (20.7%)


## 2. Poisonable questions

In [3]:
poisonable = [ex.id for ex in all_examples if find_answer_passage(ex) is not None]
known_by_any = known[MODELS[0]] | known[MODELS[1]]
print(f"Poisonable: {len(poisonable)} | of these known by at least one model: {len(set(poisonable) & known_by_any)}")

Poisonable: 3928 | of these known by at least one model: 556


## 3. Stratified sample

In [4]:
selection = stratified_sample(poisonable, known_by_any, N_KNOWN, N_UNKNOWN, seed=SEED)
selection["models"] = MODELS
json.dump(selection, open(SELECTION_PATH, "w"), indent=1)

print("Available:", selection["available"], "| selected:", len(selection["ids"]))
for m in MODELS:
    n = sum(i in known[m] for i in selection["ids"])
    print(f"{m}: known questions in the selection: {n}")

Available: {'known': 556, 'unknown': 3372} | selected: 800
llama3.1:8b: known questions in the selection: 309
qwen2.5:7b: known questions in the selection: 227
